[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naviden/Introduction-to-XAI/blob/main/Tools/ANCHOR_examples/Anchor_tabular.ipynb)

[Colab setup: dependencies and repository files](https://github.com/Naviden/Introduction-to-XAI/blob/main/docs/REPRODUCIBILITY.md#google-colab)

# Anchors for text classification

Adapted from the [official Anchor text tutorial](https://github.com/marcotcr/anchor/blob/master/notebooks/Anchor%20for%20text.ipynb). The filename is retained for existing links; this notebook explains **text**, not tabular data.

External inputs are pinned and checked against the repository [asset manifest](../../assets/manifest.json). Run from this checkout; see [download and offline guidance](../../docs/ASSETS.md).

## Setup

Install the repository dependencies before starting the notebook. The first explanation uses spaCy tokenization only, so no downloaded spaCy language model is needed. The later BERT examples additionally use PyTorch and Transformers from `requirements.txt` and need internet access on their first run to download `distilbert-base-cased`. BERT sampling can take several minutes on a CPU.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "assets/manifest.json").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from xai_assets import asset_file, hf_model_path, verify_dataset, load_nhanesi, load_imagenet50, load_adult_dataframe

from pathlib import Path
import time
import numpy as np
import sklearn.model_selection
import sklearn.linear_model
import sklearn.metrics
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from anchor import anchor_text
from IPython.display import HTML, display

In [ ]:
np.random.seed(42)

In [ ]:
# The bundled Cornell polarity corpus uses Latin-1 encoding.
def load_polarity(path=None):
    if path is None:
        # Support kernels started in this directory or at the repository root.
        candidates = [Path("rt-polaritydata"), Path("Tools/ANCHOR_examples/rt-polaritydata")]
        path = next((p for p in candidates if p.is_dir()), None)
        if path is None:
            raise FileNotFoundError("Run from the repository root or Tools/ANCHOR_examples.")
    from xai_assets import manifest, verify_file
    for key in ("polarity_negative", "polarity_positive"):
        asset = manifest()["bundled"][key]
        verify_file(Path(path) / Path(asset["path"]).name, asset["sha256"])
    data, labels = [], []
    for label, filename in enumerate(["rt-polarity.neg", "rt-polarity.pos"]):
        with (Path(path) / filename).open(encoding="latin-1") as source:
            for line in source:
                text = line.strip()
                if text:
                    data.append(text)
                    labels.append(label)
    return data, labels

In [ ]:
# Anchor only uses token text and character offsets.
nlp = spacy.blank("en")

In [ ]:
data, labels = load_polarity()

In [ ]:
data[0], labels[0]

In [ ]:
train, test, train_labels, test_labels = sklearn.model_selection.train_test_split(
    data, labels, test_size=.2, random_state=42)

train, val, train_labels, val_labels = sklearn.model_selection.train_test_split(
    train, train_labels, test_size=.1, random_state=42)

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)
val_labels = np.array(val_labels)

In [ ]:
for x in data[:5]:
    print(x) 

In [ ]:
vectorizer = CountVectorizer(min_df=1)
vectorizer.fit(train)
train_vectors = vectorizer.transform(train)
test_vectors = vectorizer.transform(test)
val_vectors = vectorizer.transform(val)

In [ ]:
c = sklearn.linear_model.LogisticRegression(max_iter=1000, random_state=42)
c.fit(train_vectors, train_labels)
preds = c.predict(val_vectors)
print('Val accuracy', sklearn.metrics.accuracy_score(val_labels, preds))

In [ ]:
def predict_lr(texts):
    return c.predict(vectorizer.transform(texts))

### Explaining a prediction
use_unk_distribution=True means we will perturb examples by replacing words with UNKS

In [ ]:
explainer = anchor_text.AnchorText(nlp, ['negative', 'positive'],
                                   use_unk_distribution=True)

In [ ]:
np.random.seed(42)
text = 'This is a good book .'
pred = explainer.class_names[predict_lr([text])[0]]
alternative = explainer.class_names[1 - predict_lr([text])[0]]
print(f'Prediction: {pred}')
print(f'Alternative: {alternative}')
exp = explainer.explain_instance(text, predict_lr, threshold=0.95)

Inspect the selected anchor and its estimated precision. Precision refers to the chosen perturbation distribution, rather than a guarantee over all natural-language sentences.

In [ ]:
print('Anchor: %s' % (' AND '.join(exp.names())))
print('Precision: %.2f' % exp.precision())
print('-'*50)
print('Examples where anchor applies and model predicts %s:' % pred)
print('\n'.join([x[0] for x in exp.examples(only_same_prediction=True)]))
print('-'*50)
print('Examples where anchor applies and model predicts %s:' % alternative)
print('\n'.join([
    x[0] for x in exp.examples(partial_index=0, only_different_prediction=True)
]))

In [ ]:
exp.precision()

## Using DistilBERT

The distribution above is a bit naive, and you get a lot of sentences that don't look realistic.  
Let's use BERT to perturb the data:

In [ ]:
# Inject a checked local model instead of Anchor's implicit mutable download.
from transformers import DistilBertTokenizer, DistilBertForMaskedLM
import torch

class PinnedTextGenerator(anchor_text.TextGenerator):
    def __init__(self):
        local_model = str(hf_model_path("distilbert_anchor"))
        self.torch, self.url, self.device = torch, None, torch.device("cpu")
        self.bert_tokenizer = DistilBertTokenizer.from_pretrained(local_model, local_files_only=True)
        self.bert = DistilBertForMaskedLM.from_pretrained(
            local_model, local_files_only=True, trust_remote_code=False
        ).to(self.device)
        self.bert.eval()

explainer = anchor_text.AnchorText(nlp, ['negative', 'positive'], use_unk_distribution=True)
explainer.use_unk_distribution = False
explainer.tg = PinnedTextGenerator()

In [ ]:
np.random.seed(42)
text = 'This is a good book .'
pred = explainer.class_names[predict_lr([text])[0]]
alternative = explainer.class_names[1 - predict_lr([text])[0]]
print(f'Prediction: {pred}')

exp = explainer.explain_instance(text,
                                 predict_lr,
                                 threshold=0.95,
                                 verbose=False)

In [ ]:
print('Anchor: %s' % (' AND '.join(exp.names())))
print('Precision: %.2f' % exp.precision())
print('-'*50)
print('Examples where anchor applies and model predicts %s:' % pred)
print()
print('\n'.join([x[0] for x in exp.examples(only_same_prediction=True)]))
print('-'*50)
print('Examples where anchor applies and model predicts %s:' % alternative)
print()
print('\n'.join([x[0] for x in exp.examples(only_different_prediction=True)]))

Inspect the first condition of the selected anchor and its sampled examples. The selected words may vary with the model and sampling seed.

In [ ]:
if exp.names():
    print('Partial anchor: %s' % (' AND '.join(exp.names(0))))
    print('Precision: %.2f' % exp.precision(0))
    print()
    print('Examples where anchor applies and model predicts %s:' % pred)
    print()
    print('\n'.join(
        [x[0] for x in exp.examples(partial_index=0, only_same_prediction=True)]))
    print()
    print('Examples where anchor applies and model predicts %s:' % alternative)
    print()
    print('\n'.join([
        x[0] for x in exp.examples(partial_index=0, only_different_prediction=True)
    ]))
else:
    print('The explanation is an empty anchor; there is no partial rule.')

Note how the examples are much more realistic than just using UNKS.

We are sampling from BERT sequentially above (one mask at a time), to get sentences that are more coherent.  
If you want to do a single BERT pass per sample, you can set `onepass=True`. You may lose some coherence, but it will be much faster.

In [ ]:
np.random.seed(1)
text = 'This is a good book .'
pred = explainer.class_names[predict_lr([text])[0]]
alternative = explainer.class_names[1 - predict_lr([text])[0]]
print('Prediction: %s' % pred)
b = time.time()
exp = explainer.explain_instance(text,
                                 predict_lr,
                                 threshold=0.95,
                                 verbose=False,
                                 onepass=True)
print('Time: %s' % (time.time() - b))

In [ ]:
print('Anchor: %s' % (' AND '.join(exp.names())))
print('Precision: %.2f' % exp.precision())
print()
print('Examples where anchor applies and model predicts %s:' % pred)
print()
print('\n'.join([x[0] for x in exp.examples(only_same_prediction=True)]))
print()
print('Examples where anchor applies and model predicts %s:' % alternative)
print()
print('\n'.join([x[0] for x in exp.examples(only_different_prediction=True)]))

Inspect the first condition of the selected anchor and its sampled examples. The selected words may vary with the model and sampling seed.

In [ ]:
if exp.names():
    print('Partial anchor: %s' % (' AND '.join(exp.names(0))))
    print('Precision: %.2f' % exp.precision(0))
    print()
    print('Examples where anchor applies and model predicts %s:' % pred)
    print()
    print('\n'.join(
        [x[0] for x in exp.examples(partial_index=0, only_same_prediction=True)]))
    print()
    print('Examples where anchor applies and model predicts %s:' % alternative)
    print()
    print('\n'.join([
        x[0] for x in exp.examples(partial_index=0, only_different_prediction=True)
    ]))
else:
    print('The explanation is an empty anchor; there is no partial rule.')

## See a visualization of the anchor with examples and etc (won't work if you're seeing this on github)

In [ ]:
# Use the current public IPython display API.
display(HTML(exp.as_html()))